<a href="https://colab.research.google.com/github/ramyars466/trainite-prototype/blob/main/trainite_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytorch-ignite


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 8.9 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from ignite.engine import Engine, Events
from ignite.handlers import ModelCheckpoint, global_step_from_engine
from ignite.handlers.tensorboard_logger import (
    TensorboardLogger, OutputHandler, OptimizerParamsHandler
)
import math
import os


In [13]:
config = {
    # Dataset
    "vocab": list("abcdefghijklmnopqrstuvwxyz"),
    "seq_len": 6,
    "dataset_size": 5000,

    # Model
    "embed_dim": 128,   # ← increased from 64
    "num_heads": 4,
    "num_layers": 3,    # ← increased from 2
    "dropout": 0.1,

    # Training
    "batch_size": 64,
    "lr": 3e-4,
    "max_epochs": 80,   # ← increased from 30

    # Output
    "output_dir": "output",
}


In [14]:
import random

# ── Dataset (deterministic, in RAM) ──
class StringReversalDataset(Dataset):
    def __init__(self, vocab, seq_len, size, seed=42):
        self.vocab = vocab
        self.seq_len = seq_len
        self.char2idx = {c: i+1 for i, c in enumerate(vocab)}  # 0 = PAD
        self.idx2char = {i+1: c for i, c in enumerate(vocab)}
        self.idx2char[0] = "<PAD>"

        # Deterministic dataset in RAM
        random.seed(seed)
        self.data = []
        for _ in range(size):
            seq = random.choices(vocab, k=seq_len)
            self.data.append(seq)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        reversed_seq = seq[::-1]
        src = torch.tensor([self.char2idx[c] for c in seq])
        tgt = torch.tensor([self.char2idx[c] for c in reversed_seq])
        return src, tgt

    def decode(self, indices):
        return "".join([self.idx2char.get(i.item(), "?") for i in indices])


# Create datasets
dataset = StringReversalDataset(
    config["vocab"], config["seq_len"], config["dataset_size"]
)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config["batch_size"])

# Quick test
src, tgt = dataset[0]
print(f"Input:   {dataset.decode(src)}")
print(f"Target:  {dataset.decode(tgt)}")
print(f"Train size: {train_size}, Val size: {val_size}")


Input:   qahftr
Target:  rtfhaq
Train size: 4000, Val size: 1000


In [15]:
# ── Decoder-Only Transformer Model ──
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2) * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = PositionalEncoding(embed_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)
        self.embed_dim = embed_dim

    def forward(self, src, tgt):
        src_emb = self.pos_encoding(self.embedding(src) * math.sqrt(self.embed_dim))
        tgt_emb = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.embed_dim))

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out = self.decoder(tgt_emb, src_emb, tgt_mask=tgt_mask)
        return self.fc_out(out)


# Initialize model
vocab_size = len(config["vocab"]) + 1  # +1 for PAD
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    embed_dim=config["embed_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    dropout=config["dropout"]
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model initialized ✅")
print(f"Vocab size: {vocab_size}")
print(f"Total parameters: {total_params:,}")


Model initialized ✅
Vocab size: 27
Total parameters: 1,984,923


In [17]:
# ── Ignite Trainer + Evaluator + Handlers ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
criterion = nn.CrossEntropyLoss(ignore_index=0)

os.makedirs(config["output_dir"], exist_ok=True)

# ── Train Step ──
def train_step(engine, batch):
    model.train()
    src, tgt = [x.to(device) for x in batch]
    tgt_input = tgt[:, :-1]
    tgt_label = tgt[:, 1:]
    optimizer.zero_grad()
    output = model(src, tgt_input)
    loss = criterion(output.reshape(-1, vocab_size), tgt_label.reshape(-1))
    loss.backward()
    optimizer.step()
    return loss.item()

# ── Eval Step ──
def eval_step(engine, batch):
    model.eval()
    with torch.no_grad():
        src, tgt = [x.to(device) for x in batch]
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]
        output = model(src, tgt_input)
        loss = criterion(output.reshape(-1, vocab_size), tgt_label.reshape(-1))
    return loss.item()

# ── Engines ──
trainer = Engine(train_step)
evaluator = Engine(eval_step)

# ── Handlers ──
@trainer.on(Events.EPOCH_COMPLETED)
def log_training(engine):
    print(f"Epoch {engine.state.epoch:3d} | Train Loss: {engine.state.output:.4f}")

@trainer.on(Events.EPOCH_COMPLETED)
def run_evaluator(engine):
    evaluator.run(val_loader)

@evaluator.on(Events.COMPLETED)
def log_validation(engine):
    print(f"           | Val   Loss: {engine.state.output:.4f}")

# Checkpoint — save best model
checkpoint = ModelCheckpoint(
    config["output_dir"],
    filename_prefix="best",
    n_saved=1,
    score_function=lambda e: -evaluator.state.output,
    score_name="val_loss",
    global_step_transform=global_step_from_engine(trainer),
    require_empty=False    # ← FIXED!
)
evaluator.add_event_handler(Events.COMPLETED, checkpoint, {"model": model})

print("Trainer ready ✅")
print(f"Device: {device}")


Trainer ready ✅
Device: cpu


In [18]:
# ── Run Training ──
print("Starting training...")
print("="*50)

trainer.run(train_loader, max_epochs=config["max_epochs"])

print("="*50)
print("Training complete! ✅")


Starting training...
Epoch   1 | Train Loss: 2.6741
           | Val   Loss: 2.5608
Epoch   2 | Train Loss: 2.5348
           | Val   Loss: 2.4342
Epoch   3 | Train Loss: 2.5637
           | Val   Loss: 2.3176
Epoch   4 | Train Loss: 2.3943
           | Val   Loss: 2.2327
Epoch   5 | Train Loss: 2.3945
           | Val   Loss: 2.2170
Epoch   6 | Train Loss: 2.3467
           | Val   Loss: 2.2023
Epoch   7 | Train Loss: 2.1790
           | Val   Loss: 2.1807
Epoch   8 | Train Loss: 2.3068
           | Val   Loss: 2.1963
Epoch   9 | Train Loss: 2.1355
           | Val   Loss: 2.0454
Epoch  10 | Train Loss: 2.0624
           | Val   Loss: 2.0220
Epoch  11 | Train Loss: 1.9534
           | Val   Loss: 2.0112
Epoch  12 | Train Loss: 1.9792
           | Val   Loss: 2.0231
Epoch  13 | Train Loss: 1.9270
           | Val   Loss: 1.9371
Epoch  14 | Train Loss: 1.9348
           | Val   Loss: 1.9614
Epoch  15 | Train Loss: 1.8747
           | Val   Loss: 1.9443
Epoch  16 | Train Loss: 2.0464
   

In [19]:
# ── Test: model("abcdef") → "fedcba" ──
def predict(model, input_str, dataset, device, max_len=6):
    model.eval()
    with torch.no_grad():
        src = torch.tensor(
            [dataset.char2idx[c] for c in input_str]
        ).unsqueeze(0).to(device)

        tgt = torch.tensor([[dataset.char2idx[input_str[-1]]]]).to(device)

        for _ in range(max_len - 1):
            output = model(src, tgt)
            next_token = output[:, -1, :].argmax(dim=-1, keepdim=True)
            tgt = torch.cat([tgt, next_token], dim=1)

        return dataset.decode(tgt[0])

# ── Run Tests (only lowercase a-z strings!) ──
print("="*40)
print("   MODEL TEST RESULTS")
print("="*40)

test_strings = ["abcdef", "ramyar", "ignite", "python", "hellow", "zxywvu"]
for s in test_strings:
    s_clean = s[:config["seq_len"]]
    predicted = predict(model, s_clean, dataset, device)
    expected = s_clean[::-1]
    status = "✅" if predicted == expected else "❌"
    print(f"{status} Input: {s_clean:8s} | Expected: {expected:8s} | Got: {predicted}")

print("="*40)


   MODEL TEST RESULTS
✅ Input: abcdef   | Expected: fedcba   | Got: fedcba
❌ Input: ramyar   | Expected: raymar   | Got: raymra
✅ Input: ignite   | Expected: etingi   | Got: etingi
✅ Input: python   | Expected: nohtyp   | Got: nohtyp
❌ Input: hellow   | Expected: wolleh   | Got: wolhew
✅ Input: zxywvu   | Expected: uvwyxz   | Got: uvwyxz
